In [2]:
from os import scandir
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('RDD Basics').getOrCreate()
sc= spark.sparkContext
print("Spark session created...!")

Spark session created...!


In [3]:
numbers=[10,20,30,40,50]
rdd= sc.parallelize(numbers)
print(rdd)

ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:297


In [4]:
rdd.collect()

[10, 20, 30, 40, 50]

In [5]:
data=["Rahul","Sneha","Arjun","Priya"]
name_rdd=sc.parallelize(data)
name_rdd.collect()

['Rahul', 'Sneha', 'Arjun', 'Priya']

In [6]:
rdd.getNumPartitions()
# spark takes control and decide how to partition

2

In [7]:
rdd=sc.parallelize(numbers,3) # manual partition control
rdd.getNumPartitions()

3

In [8]:
rdd=sc.parallelize([10,20,30,40,50,60],3)
rdd.glom().collect()

[[10, 20], [30, 40], [50, 60]]

1. **Transformations**
- These create a new RDD.
Examples:
map,
filter,
flatMap,
distinct,
union,
intersection,
groupByKey,
reduceByKey

2. **Actions**
- These return a result to the driver.
Examples:
collect,
count,
first,
take,
reduce,

In [11]:
rdd= sc.parallelize([1,2,3,4,5])
mapped_rdd=rdd.map(lambda x:x*2)
filtered_rdd=mapped_rdd.filter(lambda x:x>2)
print("Nothing executes")
#these r tranformation, will not print o/p

Nothing executes


In [12]:
print(mapped_rdd.collect())
print(filtered_rdd.collect())
#collect is an action

[2, 4, 6, 8, 10]
[4, 6, 8, 10]


**Lazy evaluation** is a key concept in Apache Spark, where the transformations on data are not immediately executed, but rather their execution is delayed until an action is triggered.

**=> Map**
is a transformation

In [13]:
rdd=sc.parallelize([1,2,3,4,5])
squared_rdd=rdd.map(lambda x:x*x)
squared_rdd.collect()

[1, 4, 9, 16, 25]

In [14]:
rdd=sc.parallelize(["Rahul","Sneha","Arjun"])
upper_rdd= rdd.map(lambda x:x.upper())
upper_rdd.collect()

['RAHUL', 'SNEHA', 'ARJUN']

**=> Filter**
is an action

In [15]:
rdd=sc.parallelize([1,2,3,4,5,6,7,8,9,10])
even_rdd=rdd.filter(lambda x:x%2==0)
even_rdd.collect()

[2, 4, 6, 8, 10]

In [16]:
rdd=sc.parallelize(["Apple","Banana","Cherry","Date","Elderberry"])
filtered_rdd=rdd.filter(lambda x:len(x)>5)
filtered_rdd.collect()

['Banana', 'Cherry', 'Elderberry']

**=> FlatMap** will return multiple elements

In [18]:
lines=sc.parallelize([
    "python spark hadoop",
    "spark sql pysqark"
])

words=lines.flatMap(lambda line:line.split(" "))
words.collect()

['python', 'spark', 'hadoop', 'spark', 'sql', 'pysqark']

**=> map vs flatMap**

In [19]:
lines=sc.parallelize(["hello world","pyspark rdd"])
result=lines.map(lambda x:x.split(" "))
result.collect()

[['hello', 'world'], ['pyspark', 'rdd']]

In [20]:
lines=sc.parallelize(["hello world","pyspark rdd"])
result=lines.flatMap(lambda x:x.split(" "))
result.collect()

['hello', 'world', 'pyspark', 'rdd']

**=> distinct**

In [21]:
rdd=sc.parallelize([1,2,2,3,4,4,5])
distinct_rdd=rdd.distinct()
distinct_rdd.collect()

[2, 4, 1, 3, 5]

**=> union**

In [22]:
rdd1=sc.parallelize([1,2,3,4,5])
rdd2=sc.parallelize([4,5,6,7,8])
union_rdd=rdd1.union(rdd2)
union_rdd.collect()

[1, 2, 3, 4, 5, 4, 5, 6, 7, 8]

**=> intersection**

In [23]:
rdd1=sc.parallelize([1,2,3,4,5])
rdd2=sc.parallelize([4,5,6,7,8])
intersection_rdd=rdd1.intersection(rdd2)
intersection_rdd.collect()

[4, 5]

**=> subtract**

In [24]:
rdd1=sc.parallelize([1,2,3,4,5])
rdd2=sc.parallelize([4,5,6,7,8])
subtract_rdd=rdd1.subtract(rdd2)
subtract_rdd.collect()

[1, 2, 3]

##**Actions**

**=> collect**

In [25]:
rdd=sc.parallelize([10,20,30])
rdd.collect()

[10, 20, 30]

**=> count**

In [26]:
rdd=sc.parallelize([10,20,30])
rdd.count()

3

**=> first**

In [27]:
rdd=sc.parallelize([10,20,30])
rdd.first()

10

**=> take**

In [28]:
rdd=sc.parallelize([10,20,30])
rdd.take(2)

[10, 20]

**=> reduce**


In [29]:
rdd=sc.parallelize([10,20,30])
rdd.reduce(lambda x,y:x+y) # reduce to a single number

60

##**Key-Value pairs** in RDD

In [31]:
data=[
    ("Apple",10),
    ("Orange",20),
    ("Banana",30),
    ("Apple",40),
    ("Orange",50)
]
rdd=sc.parallelize(data)
rdd.collect()

[('Apple', 10), ('Orange', 20), ('Banana', 30), ('Apple', 40), ('Orange', 50)]

**=> reduceByKey**

In [32]:
rdd.reduceByKey(lambda x,y:x+y).collect()
# give cumulative outcome

[('Orange', 70), ('Apple', 50), ('Banana', 30)]

**=> groupByKey**

In [33]:
grouped=rdd.groupByKey()
[(k,list(v)) for k,v in grouped.collect()]
# split based on user requirement

[('Orange', [20, 50]), ('Apple', [10, 40]), ('Banana', [30])]

**=> mapValues**

In [34]:
rdd=sc.parallelize([
    ("Apple",10),
    ("Orange",20),
    ("Banana",30),
    ("Apple",40),
    ("Orange",50)
])
rdd.mapValues(lambda x:x+5).collect()

[('Apple', 15), ('Orange', 25), ('Banana', 35), ('Apple', 45), ('Orange', 55)]

**=> sortBy**

In [35]:
rdd=sc.parallelize([10,40,20,30])
rdd.sortBy(lambda x:x).collect()

[10, 20, 30, 40]

**=> word count**

In [36]:
lines=sc.parallelize([
    "python spark hadoop",
    "spark sql pyspark"
])
words=lines.flatMap(lambda line:line.split(" "))
word_count=words.map(lambda word:(word,1))
word_count.reduceByKey(lambda x,y:x+y).collect()

[('python', 1), ('hadoop', 1), ('spark', 2), ('sql', 1), ('pyspark', 1)]

In [38]:
employees=sc.parallelize(
    [
        (1,"Rahul",18000),
        (2,"Sneha",25000),
        (3,"Arjun",30000)
    ]
)
high_sal=employees.filter(lambda x:x[2]>20000)
high_sal.collect()

[(2, 'Sneha', 25000), (3, 'Arjun', 30000)]